In this assignment, you will implement Sequence-to-Sequence (Seq2Seq) workflow for text summarization using the CNN/DailyMail dataset.    

1) Establish Baselines: Quantify the 'Context Bottleneck' using Lead-3 heuristics.

2) Fine-Tuning with Attention: Adapt a pre-trained BART or T5 model to the news domain using the Trainer API.

3) Interpretability Study: Visualize Attention Heatmaps to prove that the model can 'attend' to relevant facts regardless of their position in the text

This assignment builds on Module 4 (RNNs & LSTMs) and emphasizes the transition from manual architectural builds to industry-standard transfer learning

### Bridging the Gap: From Keras to Hugging Face

In Week 4, you built models by "wiring" layers together manually using Keras. You defined the input, hidden layers, and output, and then you told the model how to learn using `.compile()` and `.fit()`.

This week, we are moving to **Transfer Learning** using the Hugging Face `Trainer` API. This can feel like a jump in complexity because the "training loop" is no longer visible—it is abstracted away. Use the mapping below to translate what you know (Keras) to what you are about to use (Hugging Face).


* **The Model:** In Keras, you built the model step-by-step. In Hugging Face, you download a pretrained model (`AutoModelForSeq2SeqLM.from_pretrained`)

* **Configuration:** Instead of `model.compile()`, you set hyperparameters in a config object called `Seq2SeqTrainingArguments`

* **The Loop:** Instead of `model.fit()`, you initialize a `Seq2SeqTrainer` object and call `trainer.train()`. The Trainer automatically handles dynamic padding, learning rate decay, and GPU optimization.

## Part 1: Preprocessing and Dataset Preparation (20 Points)



####

Use the code below to download the CNN/DailyMail dataset (3.0.0)
Standarized Samplling of 2,000 training records, 500 validation and 500 test records

Initialize a pre-trained tokenizer (e.g. AutoTokenizer) matching your selected Transformer model (Bart or T5)

Convert the processed records into a format ready for a Keras-integrated data loader




In [ ]:
## Student Code Required ##
 # Instruction: Initialize the tokenizer and determine the optimal sequence
 # lengths for articles vs. highlights.
 # You can enforce a 2,000-sample training limit to assist with execution

# ==================== Part 1: Dataset & Tokenization ====================
from transformers import AutoTokenizer

model_checkpoint = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

def preprocess_function(examples):
    # Category A Blanks: Define truncation and padding strategies
    model_inputs = tokenizer(examples["article"],
                             max_length=____, # Choose 512 for articles
                             truncation=____,
                             padding="____")

    with tokenizer.as_target_tokenizer():
        labels = tokenizer(examples["highlights"],
                           max_length=____, # Choose 128 for summaries
                           truncation=____,
                           padding="____")

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Student Task: Map the preprocessing function to the standardized subsets
# (2000 train / 500 val / 500 test)

Exercise:

1.   In the code, we set the max sequence length for articles to 512. Why don't we set it to 2,000 to capture the whole article? Explain how the Self-Attention mechanism's memory requirement scales with sequence length (Hint: is it linear or quadratic?).

2.   Discuss how excessive truncation might lead to a loss of the 'Lead-3' information



## Part 2: Baseline Performance Benchmarking (20 Points)

In [ ]:
## Student Code Required ##
 # Instruction: Implement the Lead-3 heuristic.
 # Then, evaluate both Lead-3 and a Zero-Shot DistilBART model to
 # establish a quantitative 'floor' for performance.


 # ==================== Part 2: Baseline Comparison ====================
def get_lead_3_baseline(text):
    # Category A Blank: Implement sentence splitting logic
    sentences = text.split(____)
    return ". ".join(sentences[:3])

# Student Task: Calculate ROUGE-L for Lead-3 and Zero-Shot
# Use the evaluate_summaries() function provided in the scaffolding.

Exercise:

1.   Explain the 'Context Bottleneck.' Why do standard Seq2Seq models struggle to summarize articles when the most relevant information is buried in the middle of the text rather than the beginning?

## Part 3: Transformer Fine-Tuning with Attention (40 Points)

In [ ]:
## Student Code Required ##
 # Instruction: Configure the Seq2SeqTrainingArguments.
 # You must select the learning rate and epoch count that balances
 # convergence with the risk of catastrophic forgetting.

# ==================== Part 3: Fine-Tuning ====================
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./results",
    learning_rate=____,        # Hint: 5e-5 is standard for BART
    num_train_epochs=____,     # Hint: 3 to 5 is sufficient
    per_device_train_batch_size=8,
    predict_with_generate=True,
    fp16=torch.cuda.is_available(),
)

# Category B Scaffold: Initialization of the Trainer API
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

Exercise:

1.   How does the predict_with_generate argument differ from standard training?

2.   We configured the model to use **Beam Search** for generating summaries. Contrast Beam Search with **Greedy Search**. Why is Greedy Search highly likely to fail in a sequence-to-sequence summarization task, and how does Beam Search fix this?

## Part 4: Model Evaluation & Interpretability (20 points)

In [ ]:
## Student Code Required ##
 # Instruction: Generate a comparison table of ROUGE scores.
 # Then, execute the Heatmap visualization for ONE article and
 # identify the 'Alignment' between summary tokens and input tokens


 # ==================== Part 4: Attention Visualization ====================
# Student Task: Use aggregate_cross_attention() to visualize weights.

# Qualitative Analysis Blank:
# "In my heatmap, when the model generated the word '____', it was
# specifically attending to the input tokens '____' and '____'."

Exercise:

**1. The Mechanism:**
Describe how the Attention Mechanism you visualized allows the decoder to "bypass" the bottleneck of a single fixed context vector used in standard RNNs.

**2. Success Verification:**
In your heatmap above, identify a specific word in the summary that strongly attends to a relevant word in the source text. (e.g., "The summary word 'Apple' attended strongly to 'Cupertino' in the source").

---

**It is easy to find examples where the model works. As a Data Scientist, your job is to find where it breaks.**

**Task: Failure Mode Analysis**
Run the visualization code above for a few different samples (change the index `i`) until you find an instance of **Attention Failure** or **Hallucination**.

**Analysis Prompt:**
Locate a summary where the model generated a word that was **not** supported by the source text, or where it missed a key fact.
* **The Hallucination:** What word did the model generate that wasn't in the source?
* **The Heatmap:** Look at the attention column for that specific hallucinated token.
    * Did it attend to an irrelevant word (Misguided Attention)?
    * Was the attention "scattered" (uniform faint weight across all words)?
* **Implication:** Discuss what this failure implies for using such models in high-stakes fields like Financial News or Medical Summarization.
